# Notebook 15: How to Evaluate Agents

**Frontier ML Interview Prep** | Agentic Systems Track

---

Evaluating agents is fundamentally harder than evaluating language models. An LLM generates text; you compare it to a reference. An agent takes actions in the world, uses tools, makes plans, recovers from errors, and arrives at outcomes through diverse paths. This notebook gives you a complete framework for agent evaluation: the metrics, the benchmarks, the harness, and the cost-quality tradeoffs that come up in every frontier lab interview.

## 1. Self-Quiz (Active Recall)

Before reading anything, try to answer these from memory. Write your answers in the cell below, then check them against the notebook content.

1. **How do you measure if an agent is "good"?** What dimensions matter beyond task completion?
2. **Name 3 agent benchmarks.** What does each measure?
3. **What is the difference between task completion and process quality?** Why does the distinction matter for safety?
4. **How do you handle non-determinism** in agent evaluation? If you run the same agent twice, you may get different results.
5. **What is the cost-quality tradeoff?** When is a more expensive agent preferable to a cheaper one?

In [ ]:
# YOUR ANSWERS (write before reading the notebook):
#
# 1. How do you measure if an agent is good?
#    ...
#
# 2. Three agent benchmarks:
#    ...
#
# 3. Task completion vs process quality:
#    ...
#
# 4. Handling non-determinism:
#    ...
#
# 5. Cost-quality tradeoff:
#    ...

## 2. Setup

In [ ]:
!pip install openai matplotlib pandas numpy -q

In [ ]:
import os
import json
import time
import random
import re
import hashlib
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple, Callable
from enum import Enum
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np

# Set your OpenAI API key
# os.environ["OPENAI_API_KEY"] = "sk-..."

try:
    from openai import OpenAI
    client = OpenAI()
    USE_LLM = True
    print("OpenAI client initialized. Will use real LLM calls.")
except Exception:
    USE_LLM = False
    print("No OpenAI API key found. Using simulated agent traces for demonstration.")

print("Setup complete.")

## 3. Why Agent Eval is Hard

Evaluating language models is (relatively) straightforward: give a prompt, get text, compare to a reference or use a judge model. Agent evaluation breaks every one of these assumptions.

### Problem 1: Multi-Step Processes
An LLM produces a single output. An agent produces a **trajectory** -- a sequence of thoughts, tool calls, observations, and decisions. Two agents can solve the same task with completely different trajectories. Which one was "better"?

### Problem 2: Multiple Valid Solutions
Ask an agent to find the population of France. It could:
- Search Wikipedia directly
- Search Google, then click a result
- Use a knowledge base API
- Ask a sub-agent to research it

All valid. But some are more efficient, more reliable, or more generalizable than others.

### Problem 3: Non-Determinism and Side Effects
Agents interact with external tools. A web search today returns different results than tomorrow. A database query might time out. An API might rate-limit. The same agent with the same task can succeed one run and fail the next. You need **statistical evaluation** -- multiple runs per task.

### Problem 4: Cost Matters
A correct but expensive agent may be **worse** than a slightly less accurate cheap one. If Agent A solves 95% of tasks using 100K tokens each, and Agent B solves 90% using 10K tokens each, Agent B might be preferable in production. You must evaluate the **Pareto frontier** of cost vs quality.

### Problem 5: Safety and Process Quality
Task completion alone is dangerous. An agent that solves a coding task by deleting all test files and writing `return True` has 100% "task completion" but catastrophic process quality. For production agents, **how** they solve tasks matters as much as **whether** they solve them.

---

> **Interview Framing**: "Agent evaluation is fundamentally a multi-objective optimization problem. You're balancing task completion, process quality, cost, latency, and safety. In a production multi-agent RCA (root-cause-analysis) system, we evaluated RCA agents on both diagnostic accuracy AND the quality of the reasoning chain -- because an incorrect but well-reasoned diagnosis is more useful than a lucky guess."

## 4. Metrics Framework

Let's build a comprehensive metrics framework for agent evaluation. We'll implement each metric with clear measurement logic.

In [ ]:
# First, define our data structures for agent traces

@dataclass
class ToolCall:
    """A single tool call made by an agent."""
    tool_name: str
    arguments: Dict[str, Any]
    result: Optional[str] = None
    success: bool = True
    tokens_used: int = 0
    latency_ms: float = 0.0

@dataclass
class AgentStep:
    """One step in an agent's trajectory."""
    thought: str
    tool_call: Optional[ToolCall] = None
    observation: str = ""
    tokens_used: int = 0
    timestamp: float = 0.0

@dataclass
class AgentTrace:
    """Complete trace of an agent solving a task."""
    task_id: str
    steps: List[AgentStep] = field(default_factory=list)
    final_answer: str = ""
    completed: bool = False
    start_time: float = 0.0
    end_time: float = 0.0

    @property
    def total_tokens(self) -> int:
        return sum(s.tokens_used for s in self.steps)

    @property
    def total_tool_calls(self) -> int:
        return sum(1 for s in self.steps if s.tool_call is not None)

    @property
    def wall_clock_time(self) -> float:
        return self.end_time - self.start_time

@dataclass
class GoldStandard:
    """Gold standard for a task: expected answer and expected tool usage."""
    task_id: str
    expected_answer: str
    required_tools: List[str]  # tools that MUST be called
    forbidden_tools: List[str] = field(default_factory=list)  # tools that must NOT be called
    max_steps: int = 10  # optimal number of steps
    answer_check: str = "exact"  # "exact", "contains", "numeric_close"

print("Data structures defined.")
print(f"  ToolCall: represents a single tool invocation")
print(f"  AgentStep: one thought-action-observation triple")
print(f"  AgentTrace: full trajectory for one task")
print(f"  GoldStandard: expected answer + tool usage")

In [ ]:
class AgentMetrics:
    """
    Comprehensive metrics for agent evaluation.
    
    Six core metrics:
    1. Task completion rate: did the agent produce the correct answer?
    2. Tool use accuracy: did it call the right tools with correct arguments?
    3. Planning efficiency: how many steps to solve? (fewer = better)
    4. Cost: total tokens used (proxy for API cost)
    5. Error recovery rate: when a tool failed, did the agent recover?
    6. Time to completion: wall clock time
    """

    @staticmethod
    def task_completion_rate(
        traces: List[AgentTrace],
        golds: List[GoldStandard]
    ) -> Dict[str, Any]:
        """Binary: did the agent produce the correct final answer?"""
        gold_map = {g.task_id: g for g in golds}
        results = []

        for trace in traces:
            gold = gold_map.get(trace.task_id)
            if gold is None:
                continue

            if gold.answer_check == "exact":
                correct = trace.final_answer.strip().lower() == gold.expected_answer.strip().lower()
            elif gold.answer_check == "contains":
                correct = gold.expected_answer.strip().lower() in trace.final_answer.strip().lower()
            elif gold.answer_check == "numeric_close":
                try:
                    pred_num = float(re.findall(r'[\d.]+', trace.final_answer)[0])
                    gold_num = float(re.findall(r'[\d.]+', gold.expected_answer)[0])
                    correct = abs(pred_num - gold_num) / max(abs(gold_num), 1e-9) < 0.05
                except (ValueError, IndexError):
                    correct = False
            else:
                correct = False

            results.append({
                "task_id": trace.task_id,
                "correct": correct,
                "predicted": trace.final_answer,
                "expected": gold.expected_answer
            })

        rate = sum(r["correct"] for r in results) / max(len(results), 1)
        return {"rate": rate, "correct": sum(r["correct"] for r in results),
                "total": len(results), "details": results}

    @staticmethod
    def tool_use_accuracy(
        traces: List[AgentTrace],
        golds: List[GoldStandard]
    ) -> Dict[str, Any]:
        """Did the agent call the right tools? Computes recall (the fraction of required
        tools actually used) plus forbidden-tool violations -- it does NOT compute precision."""
        gold_map = {g.task_id: g for g in golds}
        results = []

        for trace in traces:
            gold = gold_map.get(trace.task_id)
            if gold is None:
                continue

            tools_called = set(
                s.tool_call.tool_name for s in trace.steps if s.tool_call
            )
            required = set(gold.required_tools)
            forbidden = set(gold.forbidden_tools)

            # Recall: did we call all required tools?
            recall = len(required & tools_called) / max(len(required), 1)

            # Precision: of the tools we called, how many were required?
            # (tools not in required or forbidden are "neutral")
            forbidden_used = len(forbidden & tools_called)

            # Penalty for using forbidden tools
            penalty = forbidden_used > 0

            results.append({
                "task_id": trace.task_id,
                "recall": recall,
                "forbidden_violations": forbidden_used,
                "tools_called": list(tools_called),
                "required_tools": gold.required_tools
            })

        avg_recall = np.mean([r["recall"] for r in results]) if results else 0
        total_violations = sum(r["forbidden_violations"] for r in results)
        return {"avg_recall": avg_recall, "total_forbidden_violations": total_violations,
                "details": results}

    @staticmethod
    def planning_efficiency(
        traces: List[AgentTrace],
        golds: List[GoldStandard]
    ) -> Dict[str, Any]:
        """How many steps did the agent take vs the optimal? Lower ratio = better."""
        gold_map = {g.task_id: g for g in golds}
        results = []

        for trace in traces:
            gold = gold_map.get(trace.task_id)
            if gold is None:
                continue

            actual_steps = len(trace.steps)
            optimal_steps = gold.max_steps
            # Efficiency ratio: 1.0 = optimal, >1.0 = took more steps
            ratio = actual_steps / max(optimal_steps, 1)

            results.append({
                "task_id": trace.task_id,
                "actual_steps": actual_steps,
                "optimal_steps": optimal_steps,
                "efficiency_ratio": ratio
            })

        avg_ratio = np.mean([r["efficiency_ratio"] for r in results]) if results else 0
        return {"avg_efficiency_ratio": avg_ratio, "details": results}

    @staticmethod
    def cost(
        traces: List[AgentTrace]
    ) -> Dict[str, Any]:
        """Total and per-task token usage as a proxy for API cost."""
        results = []
        for trace in traces:
            results.append({
                "task_id": trace.task_id,
                "total_tokens": trace.total_tokens,
                "num_tool_calls": trace.total_tool_calls
            })

        total_tokens = sum(r["total_tokens"] for r in results)
        avg_tokens = total_tokens / max(len(results), 1)
        # Rough cost estimate at $0.01 per 1K tokens (blended input/output)
        est_cost = total_tokens / 1000 * 0.01
        return {"total_tokens": total_tokens, "avg_tokens_per_task": avg_tokens,
                "estimated_cost_usd": est_cost, "details": results}

    @staticmethod
    def error_recovery_rate(
        traces: List[AgentTrace]
    ) -> Dict[str, Any]:
        """When a tool call failed, did the agent recover and eventually complete the task?"""
        results = []
        for trace in traces:
            failures = [s for s in trace.steps if s.tool_call and not s.tool_call.success]
            if not failures:
                continue  # no failures to recover from

            # Did the agent complete the task despite tool failures?
            recovered = trace.completed
            results.append({
                "task_id": trace.task_id,
                "num_failures": len(failures),
                "recovered": recovered
            })

        recovery_rate = (
            sum(r["recovered"] for r in results) / max(len(results), 1)
            if results else None
        )
        return {"recovery_rate": recovery_rate, "tasks_with_failures": len(results),
                "details": results}

    @staticmethod
    def time_to_completion(
        traces: List[AgentTrace]
    ) -> Dict[str, Any]:
        """Wall clock time for each task."""
        results = []
        for trace in traces:
            results.append({
                "task_id": trace.task_id,
                "wall_clock_s": trace.wall_clock_time,
                "completed": trace.completed
            })

        times = [r["wall_clock_s"] for r in results if r["completed"]]
        avg_time = np.mean(times) if times else 0
        return {"avg_time_s": avg_time, "details": results}

print("AgentMetrics class defined with 6 core metrics:")
print("  1. task_completion_rate  - correct answer?")
print("  2. tool_use_accuracy     - right tools called?")
print("  3. planning_efficiency   - steps vs optimal")
print("  4. cost                  - token usage")
print("  5. error_recovery_rate   - handles failures?")
print("  6. time_to_completion    - wall clock time")

### Why These Six Metrics?

| Metric | What It Captures | Why It Matters |
|--------|-----------------|----------------|
| **Task Completion** | Correctness of final answer | The bottom line -- did the agent do its job? |
| **Tool Use Accuracy** | Correct tool selection and arguments | Catches agents that get lucky despite wrong process |
| **Planning Efficiency** | Steps taken vs optimal | Efficient agents scale better and cost less |
| **Cost** | Token consumption | Direct $ impact in production |
| **Error Recovery** | Resilience to tool failures | Production tools fail; agents must handle it |
| **Time to Completion** | Latency | User experience; some tasks are time-sensitive |

A good agent scores well on ALL six, not just task completion.

## 5. Agent Benchmarks Survey

The field has developed several benchmarks to evaluate agents in different domains. Here's the landscape as of 2025-2026:

---

### SWE-bench / SWE-bench Verified
- **What it measures**: Can an agent resolve real GitHub issues by editing code in a full repository?
- **Format**: 2,294 tasks from 12 popular Python repos. Each task is a GitHub issue + the repo at the commit before the fix. Agent must produce a patch.
- **SWE-bench Verified**: 500 human-validated instances curated by OpenAI (Aug 2024), removing ambiguous/unfixable issues.
- **Score trajectory**: ~33% (late 2024) → ~70% (mid-2025) → ~80% (early 2026). For context, SWE-agent scored 12.5% on the full SWE-bench in 2024 — full-set numbers are not directly comparable to Verified. Note: OpenAI deprecated SWE-bench Verified as an internal eval in Feb 2026, citing test flaws and contamination.
- **Limitations**: Python-only, open-source repos only, some issues require domain knowledge not in the repo.
- **Key insight**: Tests end-to-end software engineering -- understanding the codebase, localizing the bug, writing the fix, making sure tests pass.

### WebArena
- **What it measures**: Can an agent complete tasks on real websites?
- **Format**: 812 tasks on self-hosted web applications (e-commerce, Reddit, GitLab, CMS, map). Agent interacts via browser actions.
- **Score trajectory**: ~42% with GPT-4-based agents was a 2024-era number; current scores are far higher — check the live leaderboard.
- **Limitations**: Tasks are relatively template-based; real web usage is more varied.
- **Key insight**: Tests navigation, form filling, multi-step web interactions.

### OSWorld
- **What it measures**: Can an agent complete tasks on real desktop operating systems?
- **Format**: 369 tasks on Ubuntu, plus a separate Windows task set (no macOS benchmark tasks). Tasks require interacting with real OS GUIs -- file management, app usage, settings configuration.
- **Notable result**: The human baseline is 72.36%; the best agents exceeded 60% on OSWorld-Verified by 2026 — check the live leaderboard for current numbers.
- **Limitations**: Setup-intensive; requires VM environments for reproducibility.
- **Key insight**: Tests grounded GUI interaction on a real OS, where agents still trail the human baseline.

### AgentBench
- **What it measures**: Multi-domain agent capabilities across 8 environments.
- **Format**: Tasks spanning web browsing, database operations, knowledge graph, digital card games, lateral thinking puzzles, operating systems, and more.
- **Current SOTA**: GPT-4-class models lead; open-source models significantly lag.
- **Limitations**: Breadth over depth; each domain has limited tasks.
- **Key insight**: Tests generalization -- can one agent architecture handle diverse task types?

### GAIA (General AI Assistants)
- **What it measures**: Multi-tool, multi-step reasoning tasks for AI assistants.
- **Format**: 466 tasks at 3 difficulty levels. Tasks require web search, file processing, coding, math -- often several in combination.
- **Score trajectory**: 2024-era numbers were ~75% on Level 1, ~55% on Level 2, ~35% on Level 3; current scores are far higher (top deep-research agents exceed 75% overall) — check the leaderboard.
- **Limitations**: Static tasks; may be memorized by future models.
- **Key insight**: Designed so that tasks are "easy for humans, hard for AI" -- testing real-world assistant capabilities.

### tau-bench
- **What it measures**: Enterprise agent tasks in retail and airline customer service.
- **Format**: 165 tasks (115 retail + 50 airline) requiring database lookups, policy adherence, multi-turn conversation, and action execution. τ²-bench (2025) adds a dual-control telecom domain where both the user and the agent act.
- **Score trajectory**: early models scored ~35-55%; current numbers are much higher — check the leaderboard.
- **Limitations**: Domain-specific; may not generalize to other enterprise settings.
- **Key insight**: Tests the full stack of production agent requirements: tool use, policy compliance, multi-turn state tracking.

### Berkeley Function-Calling Leaderboard (BFCL)
- **What it measures**: Raw tool/function calling accuracy -- can the model produce correct tool calls?
- **Format**: 2,000+ function calling scenarios: simple, parallel, multiple, nested, and in different languages (Python, Java, JS).
- **Current SOTA**: top models do well on single-turn categories, but BFCL V3's multi-turn categories drag overall accuracy far below 90%; V4 (July 2025) added agentic categories (web search, memory).
- **Limitations**: Tests atomic tool calling, not multi-step agent behavior.
- **Key insight**: Foundation metric -- if a model can't call tools accurately, it can't be a good agent.

---

### Benchmark Selection Guide

| If you care about... | Use this benchmark |
|---------------------|--------------------|
| Coding agents | SWE-bench Verified |
| Web navigation | WebArena |
| Desktop automation | OSWorld |
| General breadth | AgentBench |
| Multi-tool reasoning | GAIA |
| Enterprise tasks | tau-bench |
| Raw tool calling | BFCL |

**Insider Tip:** SWE-bench is the most-cited coding-agent benchmark. Verified scores went from ~33% (late 2024) to ~70% (mid-2025) to ~80% (early 2026), and OpenAI deprecated it as an internal eval in Feb 2026 (test flaws + contamination). If asked "how good are agents?", this trajectory grounds the discussion. Also mention: agents remain stronger at narrow tasks (single-turn function calling) than open-ended ones (web/desktop navigation), though the gap narrowed sharply through 2025-2026.

**Insider Tip:** Be skeptical of agent benchmarks in interviews. Many are easily gamed (e.g., SWE-bench has known contamination concerns). OSWorld is more robust because it uses real desktop environments. Showing this critical thinking impresses interviewers.

## 6. Build a Mini Evaluation Harness

Now let's build a complete evaluation harness from scratch. We'll define tasks, run a simulated agent, and compute all metrics.

In [ ]:
# Define our tool catalog (simulated tools for evaluation)

TOOL_CATALOG = {
    "calculator": {
        "description": "Perform arithmetic calculations",
        "parameters": {"expression": "str"},
        "handler": lambda args: str(eval(args.get("expression", "0")))
    },
    "web_search": {
        "description": "Search the web for information",
        "parameters": {"query": "str"},
        "handler": lambda args: {
            "population of france":  # keys lowercase: handler lowercases the query "France has a population of approximately 67.75 million people.",
            "capital of japan": "The capital of Japan is Tokyo.",
            "gdp of usa 2024": "The GDP of the United States in 2024 was approximately $28.78 trillion.",
            "distance earth to moon": "The average distance from Earth to the Moon is 384,400 km.",
            "boiling point of water": "Water boils at 100 degrees Celsius (212 degrees Fahrenheit) at standard pressure.",
            "speed of light": "The speed of light in vacuum is approximately 299,792,458 meters per second.",
            "largest ocean": "The Pacific Ocean is the largest ocean, covering about 165.25 million square kilometers.",
            "gpt-4 capabilities": "GPT-4 is a large multimodal model with strong reasoning, coding, and vision capabilities.",
        }.get(args.get("query", "").lower(), f"No results found for: {args.get('query', '')}")
    },
    "database_query": {
        "description": "Query a SQL database",
        "parameters": {"sql": "str"},
        "handler": lambda args: {
            "SELECT COUNT(*) FROM users": "4,521",
            "SELECT AVG(revenue) FROM sales WHERE year=2024": "$1,234,567",
            "SELECT name FROM products ORDER BY sales DESC LIMIT 1": "Widget Pro X",
        }.get(args.get("sql", ""), "Query returned 0 results.")
    },
    "file_reader": {
        "description": "Read contents of a file",
        "parameters": {"path": "str"},
        "handler": lambda args: {
            "report.txt": "Q4 Revenue: $5.2M, Expenses: $3.1M, Profit: $2.1M",
            "config.json": '{"model": "gpt-4", "temperature": 0.7, "max_tokens": 4096}',
        }.get(args.get("path", ""), "File not found.")
    },
    "code_executor": {
        "description": "Execute Python code and return output",
        "parameters": {"code": "str"},
        "handler": lambda args: str(eval(args.get("code", "None")))
    },
    "email_sender": {
        "description": "Send an email (simulated)",
        "parameters": {"to": "str", "subject": "str", "body": "str"},
        "handler": lambda args: f"Email sent to {args.get('to', 'unknown')}"
    },
    "flaky_api": {
        "description": "An API that sometimes fails (for testing error recovery)",
        "parameters": {"request": "str"},
        "handler": lambda args: "API_TIMEOUT_ERROR" if random.random() < 0.6 else f"Result: {args.get('request', '')}"
    },
}

print(f"Defined {len(TOOL_CATALOG)} simulated tools:")
for name, tool in TOOL_CATALOG.items():
    print(f"  - {name}: {tool['description']}")

In [ ]:
# Define our evaluation tasks

@dataclass
class EvalTask:
    """An evaluation task with gold standard."""
    task_id: str
    description: str
    category: str  # "simple", "multi_step", "error_recovery"
    gold: GoldStandard = None


EVAL_TASKS = [
    # --- Simple lookup (1-2 tools) ---
    EvalTask(
        task_id="T01",
        description="What is the population of France?",
        category="simple",
        gold=GoldStandard(
            task_id="T01", expected_answer="67.75 million",
            required_tools=["web_search"], max_steps=2,
            answer_check="contains"
        )
    ),
    EvalTask(
        task_id="T02",
        description="What is 1547 * 382?",
        category="simple",
        gold=GoldStandard(
            task_id="T02", expected_answer="590954",
            required_tools=["calculator"], max_steps=2,
            answer_check="contains"
        )
    ),
    EvalTask(
        task_id="T03",
        description="What is the capital of Japan?",
        category="simple",
        gold=GoldStandard(
            task_id="T03", expected_answer="Tokyo",
            required_tools=["web_search"], max_steps=2,
            answer_check="contains"
        )
    ),
    EvalTask(
        task_id="T04",
        description="Read report.txt and tell me the Q4 profit.",
        category="simple",
        gold=GoldStandard(
            task_id="T04", expected_answer="$2.1M",
            required_tools=["file_reader"], max_steps=2,
            answer_check="contains"
        )
    ),

    # --- Multi-step reasoning (3-5 tools) ---
    EvalTask(
        task_id="T05",
        description="How many users are in the database, and what is that number divided by 3?",
        category="multi_step",
        gold=GoldStandard(
            task_id="T05", expected_answer="1507",
            required_tools=["database_query", "calculator"], max_steps=4,
            answer_check="contains"
        )
    ),
    EvalTask(
        task_id="T06",
        description="Search for the GDP of USA in 2024, then calculate what 5% of it is.",
        category="multi_step",
        gold=GoldStandard(
            # The calculator returns the raw float ("1.4390000000000001"), so a gold string of
            # "1.439 trillion" could never match with a "contains" check; check the numeric prefix instead.
            task_id="T06", expected_answer="1.439",
            required_tools=["web_search", "calculator"], max_steps=4,
            answer_check="contains"
        )
    ),
    EvalTask(
        task_id="T07",
        description="Read config.json, find the model name, then search the web for its capabilities.",
        category="multi_step",
        gold=GoldStandard(
            # Don't use "gpt-4" as the gold: the error echo "No results found for: gpt-4 capabilities"
            # would also contain it. "multimodal" only appears in a successful search result.
            task_id="T07", expected_answer="multimodal",
            required_tools=["file_reader", "web_search"], max_steps=4,
            answer_check="contains"
        )
    ),

    # --- Error recovery (tool intentionally fails) ---
    EvalTask(
        task_id="T08",
        description="Use the flaky_api to get the weather forecast. It may fail; keep trying.",
        category="error_recovery",
        gold=GoldStandard(
            task_id="T08", expected_answer="Result: weather forecast",
            required_tools=["flaky_api"], max_steps=5,
            answer_check="contains"
        )
    ),
    EvalTask(
        task_id="T09",
        description="Try to read nonexistent.txt. If it fails, search the web for the answer instead: what is the largest ocean?",
        category="error_recovery",
        gold=GoldStandard(
            task_id="T09", expected_answer="Pacific Ocean",
            required_tools=["file_reader", "web_search"], max_steps=4,
            answer_check="contains"
        )
    ),
    EvalTask(
        task_id="T10",
        description="Query the database for 'SELECT name FROM nonexistent_table'. If that fails, try 'SELECT name FROM products ORDER BY sales DESC LIMIT 1'.",
        category="error_recovery",
        gold=GoldStandard(
            task_id="T10", expected_answer="Widget Pro X",
            required_tools=["database_query"], max_steps=4,
            answer_check="contains"
        )
    ),
]

print(f"Defined {len(EVAL_TASKS)} evaluation tasks:")
for cat in ["simple", "multi_step", "error_recovery"]:
    tasks = [t for t in EVAL_TASKS if t.category == cat]
    print(f"  {cat}: {len(tasks)} tasks")
    for t in tasks:
        print(f"    - {t.task_id}: {t.description[:60]}...")

In [ ]:
class SimulatedReActAgent:
    """
    A simulated ReAct agent for evaluation purposes.
    Uses pre-defined strategies to solve tasks, with realistic
    success rates, error patterns, and token usage.
    """

    def __init__(self, tools: Dict, success_rate: float = 0.8, verbose: bool = False):
        self.tools = tools
        self.success_rate = success_rate
        self.verbose = verbose

    def _execute_tool(self, tool_name: str, args: Dict) -> ToolCall:
        tool = self.tools.get(tool_name)
        if tool is None:
            return ToolCall(
                tool_name=tool_name, arguments=args,
                result="Tool not found", success=False,
                tokens_used=50, latency_ms=100
            )
        try:
            result = tool["handler"](args)
            is_error = "ERROR" in str(result) or "not found" in str(result).lower()
            return ToolCall(
                tool_name=tool_name, arguments=args,
                result=str(result), success=not is_error,
                tokens_used=random.randint(100, 300),
                latency_ms=random.uniform(200, 2000)
            )
        except Exception as e:
            return ToolCall(
                tool_name=tool_name, arguments=args,
                result=str(e), success=False,
                tokens_used=50, latency_ms=100
            )

    def solve(self, task: EvalTask) -> AgentTrace:
        trace = AgentTrace(task_id=task.task_id, start_time=time.time())
        # Python's built-in str hash is salted per process (PYTHONHASHSEED), so hash()
        # is NOT stable across runs -- use a deterministic hash for reproducibility.
        random.seed(int(hashlib.md5(task.task_id.encode()).hexdigest(), 16) + int(self.success_rate * 100))

        # Strategy lookup for each task
        strategies = {
            "T01": [("web_search", {"query": "population of France"})],
            "T02": [("calculator", {"expression": "1547 * 382"})],
            "T03": [("web_search", {"query": "capital of Japan"})],
            "T04": [("file_reader", {"path": "report.txt"})],
            "T05": [("database_query", {"sql": "SELECT COUNT(*) FROM users"}),
                     ("calculator", {"expression": "4521 / 3"})],
            "T06": [("web_search", {"query": "GDP of USA 2024"}),
                     ("calculator", {"expression": "28.78 * 0.05"})],
            "T07": [("file_reader", {"path": "config.json"}),
                     ("web_search", {"query": "gpt-4 capabilities"})],
            "T08": [("flaky_api", {"request": "weather forecast"})] * 4,
            "T09": [("file_reader", {"path": "nonexistent.txt"}),
                     ("web_search", {"query": "largest ocean"})],
            "T10": [("database_query", {"sql": "SELECT name FROM nonexistent_table"}),
                     ("database_query", {"sql": "SELECT name FROM products ORDER BY sales DESC LIMIT 1"})],
        }

        plan = strategies.get(task.task_id, [])

        # Simulate the agent making mistakes sometimes
        should_make_mistake = random.random() > self.success_rate
        mistake_type = random.choice(["wrong_tool", "wrong_args", "extra_steps", "give_up"])

        last_result = ""
        for i, (tool_name, args) in enumerate(plan):
            # Introduce mistakes
            if should_make_mistake and i == 0 and mistake_type == "wrong_tool":
                tool_name = "email_sender"  # wrong tool
                args = {"to": "test@test.com", "subject": "Help", "body": task.description}
            elif should_make_mistake and i == 0 and mistake_type == "wrong_args":
                args = {"wrong_key": "wrong_value"}
            elif should_make_mistake and mistake_type == "give_up" and i > 0:
                break

            # Add extra unnecessary steps sometimes
            if should_make_mistake and mistake_type == "extra_steps" and i == 0:
                extra_call = self._execute_tool("web_search", {"query": "random search"})
                trace.steps.append(AgentStep(
                    thought="Let me search for some context first...",
                    tool_call=extra_call,
                    observation=extra_call.result or "",
                    tokens_used=random.randint(200, 500),
                    timestamp=time.time()
                ))

            tool_call = self._execute_tool(tool_name, args)
            thought = f"Step {i+1}: I need to use {tool_name} with {args}"
            trace.steps.append(AgentStep(
                thought=thought,
                tool_call=tool_call,
                observation=tool_call.result or "",
                tokens_used=random.randint(200, 500),
                timestamp=time.time()
            ))

            if tool_call.success:
                last_result = tool_call.result

            if self.verbose:
                status = "OK" if tool_call.success else "FAIL"
                print(f"  [{task.task_id}] {tool_name} -> [{status}] {tool_call.result[:50]}")

        # Generate final answer
        if last_result and not (should_make_mistake and mistake_type == "give_up"):
            trace.final_answer = last_result
            trace.completed = True
        else:
            trace.final_answer = "I was unable to complete this task."
            trace.completed = False

        trace.end_time = time.time()
        return trace

print("SimulatedReActAgent defined.")
print("This agent uses pre-defined strategies with configurable error rates.")

In [ ]:
class EvalHarness:
    """
    Complete evaluation harness for agent assessment.
    
    Runs an agent on a set of tasks, collects traces,
    computes all metrics, and generates a formatted report.
    """

    def __init__(self, tasks: List[EvalTask]):
        self.tasks = tasks
        self.golds = [t.gold for t in tasks if t.gold]
        self.traces: List[AgentTrace] = []
        self.metrics_results: Dict[str, Any] = {}

    def run_eval(self, agent, n_runs: int = 1) -> List[AgentTrace]:
        """Run the agent on all tasks, optionally multiple times."""
        all_traces = []
        for run in range(n_runs):
            for task in self.tasks:
                trace = agent.solve(task)
                all_traces.append(trace)
        self.traces = all_traces
        return all_traces

    def score(self) -> Dict[str, Any]:
        """Compute all metrics on collected traces."""
        metrics = AgentMetrics()
        self.metrics_results = {
            "task_completion": metrics.task_completion_rate(self.traces, self.golds),
            "tool_use": metrics.tool_use_accuracy(self.traces, self.golds),
            "planning": metrics.planning_efficiency(self.traces, self.golds),
            "cost": metrics.cost(self.traces),
            "error_recovery": metrics.error_recovery_rate(self.traces),
            "timing": metrics.time_to_completion(self.traces),
        }
        return self.metrics_results

    def report(self) -> str:
        """Generate a formatted results table."""
        if not self.metrics_results:
            self.score()

        m = self.metrics_results
        lines = [
            "=" * 60,
            "AGENT EVALUATION REPORT",
            "=" * 60,
            "",
            f"Tasks evaluated: {len(self.tasks)}",
            f"Total traces: {len(self.traces)}",
            "",
            "--- CORE METRICS ---",
            f"  Task Completion Rate:  {m['task_completion']['rate']:.1%}  "
            f"({m['task_completion']['correct']}/{m['task_completion']['total']})",
            f"  Tool Use Recall:       {m['tool_use']['avg_recall']:.1%}",
            f"  Forbidden Violations:  {m['tool_use']['total_forbidden_violations']}",
            f"  Planning Efficiency:   {m['planning']['avg_efficiency_ratio']:.2f}x optimal",
            f"  Total Tokens:          {m['cost']['total_tokens']:,}",
            f"  Avg Tokens/Task:       {m['cost']['avg_tokens_per_task']:,.0f}",
            f"  Estimated Cost:        ${m['cost']['estimated_cost_usd']:.4f}",
        ]

        if m['error_recovery']['recovery_rate'] is not None:
            lines.append(
                f"  Error Recovery Rate:   {m['error_recovery']['recovery_rate']:.1%}  "
                f"({m['error_recovery']['tasks_with_failures']} tasks with failures)"
            )
        lines.append(f"  Avg Time/Task:         {m['timing']['avg_time_s']:.3f}s")

        # Per-task breakdown
        lines.extend(["", "--- PER-TASK BREAKDOWN ---"])
        lines.append(f"{'Task':<6} {'Correct':<9} {'Steps':<7} {'Tokens':<8} {'Tools Used'}")
        lines.append("-" * 60)

        tc_details = {d["task_id"]: d for d in m["task_completion"]["details"]}
        tu_details = {d["task_id"]: d for d in m["tool_use"]["details"]}
        pl_details = {d["task_id"]: d for d in m["planning"]["details"]}
        co_details = {d["task_id"]: d for d in m["cost"]["details"]}

        for task in self.tasks:
            tid = task.task_id
            correct = "Yes" if tc_details.get(tid, {}).get("correct") else "No"
            steps = pl_details.get(tid, {}).get("actual_steps", "?")
            tokens = co_details.get(tid, {}).get("total_tokens", 0)
            tools = ", ".join(tu_details.get(tid, {}).get("tools_called", []))
            lines.append(f"{tid:<6} {correct:<9} {steps:<7} {tokens:<8} {tools}")

        lines.extend(["", "=" * 60])
        report_str = "\n".join(lines)
        return report_str

print("EvalHarness defined with methods: run_eval(), score(), report()")

In [ ]:
# Run the evaluation!

# Create agent with 80% success rate
agent = SimulatedReActAgent(TOOL_CATALOG, success_rate=0.80, verbose=True)

# Create harness and run
harness = EvalHarness(EVAL_TASKS)
print("Running evaluation...\n")
traces = harness.run_eval(agent, n_runs=1)

# Score and report
harness.score()
print("\n" + harness.report())

### Interpreting the Results

Key things to look at:
- **Task Completion Rate**: Overall success, but look at which categories fail
- **Planning Efficiency > 1.0**: Agent is taking more steps than needed
- **Error Recovery**: Critical for production agents -- can it handle tool failures?
- **Per-Task Breakdown**: Spot patterns -- does the agent struggle with multi-step tasks?

**Why does this matter for interviews?** You will likely be asked to *design* an evaluation for an agent system. This framework gives you a concrete starting point.

## 7. Error Analysis

When agents fail, understanding *why* they fail is more important than the failure rate itself. Let's build an error analyzer.

In [ ]:
class ErrorCategory(Enum):
    TOOL_SELECTION = "Tool Selection Error"      # Wrong tool chosen
    ARGUMENT_ERROR = "Argument Error"             # Right tool, wrong arguments
    PLANNING_ERROR = "Planning Error"             # Correct tools but wrong order/strategy
    HALLUCINATION = "Hallucination"               # Made up information
    CONTEXT_LIMIT = "Context Limit"               # Lost track of information over long traces
    TOOL_FAILURE = "Unrecovered Tool Failure"     # Tool failed and agent gave up
    INCOMPLETE = "Incomplete Execution"           # Stopped before finishing


class ErrorAnalyzer:
    """
    Analyzes agent failures and categorizes them.
    """

    def __init__(self, traces: List[AgentTrace], golds: List[GoldStandard]):
        self.traces = traces
        self.gold_map = {g.task_id: g for g in golds}
        self.errors: List[Dict[str, Any]] = []

    def analyze(self) -> List[Dict[str, Any]]:
        """Categorize all failures."""
        self.errors = []

        for trace in self.traces:
            gold = self.gold_map.get(trace.task_id)
            if gold is None:
                continue

            # Check if task was completed correctly
            if gold.answer_check == "contains":
                correct = gold.expected_answer.lower() in trace.final_answer.lower()
            else:
                correct = trace.final_answer.strip().lower() == gold.expected_answer.strip().lower()

            if correct:
                continue  # No error to analyze

            # Categorize the error
            tools_called = [
                s.tool_call.tool_name for s in trace.steps if s.tool_call
            ]
            required_tools = set(gold.required_tools)
            called_tools = set(tools_called)
            tool_failures = [
                s for s in trace.steps if s.tool_call and not s.tool_call.success
            ]

            # Determine error category
            if not trace.completed:
                if tool_failures:
                    category = ErrorCategory.TOOL_FAILURE
                else:
                    category = ErrorCategory.INCOMPLETE
            elif not required_tools.issubset(called_tools):
                # Didn't call required tools -> check if wrong tool or wrong args
                if called_tools - required_tools:  # called extra/wrong tools
                    category = ErrorCategory.TOOL_SELECTION
                else:
                    category = ErrorCategory.PLANNING_ERROR
            elif any("not found" in str(s.tool_call.result).lower()
                     for s in trace.steps if s.tool_call and s.tool_call.success):
                category = ErrorCategory.ARGUMENT_ERROR
            elif len(trace.steps) > gold.max_steps * 2:
                category = ErrorCategory.CONTEXT_LIMIT
            else:
                category = ErrorCategory.HALLUCINATION

            self.errors.append({
                "task_id": trace.task_id,
                "category": category,
                "predicted": trace.final_answer[:80],
                "expected": gold.expected_answer,
                "tools_called": tools_called,
                "required_tools": gold.required_tools,
                "num_steps": len(trace.steps),
                "had_tool_failures": len(tool_failures) > 0
            })

        return self.errors

    def summary(self) -> Dict[str, int]:
        """Count errors by category."""
        counts = {}
        for err in self.errors:
            cat = err["category"].value
            counts[cat] = counts.get(cat, 0) + 1
        return counts

    def print_report(self):
        """Print detailed error analysis."""
        if not self.errors:
            print("No errors found! All tasks completed correctly.")
            return

        print(f"\n{'='*60}")
        print(f"ERROR ANALYSIS: {len(self.errors)} failures")
        print(f"{'='*60}")

        summary = self.summary()
        print("\nError Distribution:")
        for cat, count in sorted(summary.items(), key=lambda x: -x[1]):
            bar = "#" * (count * 4)
            print(f"  {cat:<30} {count:>3}  {bar}")

        print("\nDetailed Failures:")
        for err in self.errors:
            print(f"  [{err['task_id']}] {err['category'].value}")
            print(f"    Expected: {err['expected']}")
            print(f"    Got:      {err['predicted']}")
            print(f"    Tools:    {err['tools_called']}")
            print()

print("ErrorAnalyzer defined.")

In [ ]:
# Run error analysis on our evaluation results

analyzer = ErrorAnalyzer(traces, [t.gold for t in EVAL_TASKS])
analyzer.analyze()
analyzer.print_report()

In [ ]:
# Visualize error distribution with a pie chart

summary = analyzer.summary()

if summary:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Pie chart of error types
    labels = list(summary.keys())
    sizes = list(summary.values())
    colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))
    explode = [0.05] * len(labels)

    axes[0].pie(sizes, explode=explode, labels=labels, colors=colors,
                autopct='%1.0f%%', shadow=True, startangle=90)
    axes[0].set_title('Error Type Distribution', fontsize=13, fontweight='bold')

    # Bar chart of errors by task category
    cat_errors = {"simple": 0, "multi_step": 0, "error_recovery": 0}
    cat_totals = {"simple": 0, "multi_step": 0, "error_recovery": 0}
    for task in EVAL_TASKS:
        cat_totals[task.category] += 1
        if any(e["task_id"] == task.task_id for e in analyzer.errors):
            cat_errors[task.category] += 1

    categories = list(cat_errors.keys())
    error_rates = [cat_errors[c] / max(cat_totals[c], 1) for c in categories]
    success_rates = [1 - er for er in error_rates]

    x = np.arange(len(categories))
    width = 0.35
    axes[1].bar(x - width/2, success_rates, width, label='Success', color='#2ecc71')
    axes[1].bar(x + width/2, error_rates, width, label='Failure', color='#e74c3c')
    axes[1].set_xlabel('Task Category')
    axes[1].set_ylabel('Rate')
    axes[1].set_title('Success/Failure by Task Category', fontsize=13, fontweight='bold')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(categories, rotation=15)
    axes[1].legend()
    axes[1].set_ylim(0, 1.15)
    for i, (s, e) in enumerate(zip(success_rates, error_rates)):
        axes[1].text(i - width/2, s + 0.02, f"{s:.0%}", ha='center', fontsize=9)
        axes[1].text(i + width/2, e + 0.02, f"{e:.0%}", ha='center', fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print("No errors to visualize -- all tasks passed!")

### Error Category Remediation Guide

| Error Category | Example | Fix |
|---------------|---------|-----|
| **Tool Selection** | Agent uses `email_sender` instead of `web_search` | Better tool descriptions; few-shot examples in system prompt |
| **Argument Error** | Right tool, wrong argument format | Schema validation; type hints in tool specs |
| **Planning Error** | Steps in wrong order; misses a required step | Add explicit planning step before execution; chain-of-thought |
| **Hallucination** | Agent fabricates an answer instead of using tools | Force tool use for factual questions; add verification step |
| **Context Limit** | Agent forgets earlier information in long traces | Compress context; use working memory; shorter trajectories |
| **Tool Failure** | Tool returns error, agent gives up | Retry logic; fallback tools; error handling prompts |

## 8. Cost-Quality Tradeoff

In production, you don't just want the most accurate agent -- you want the best agent *for your budget*.

In [ ]:
# Simulate agents with different cost-quality profiles

profiles = [
    {"name": "Cheap ReAct (GPT-3.5)",  "success_rate": 0.55, "token_mult": 0.5},
    {"name": "Standard ReAct (GPT-4)",  "success_rate": 0.75, "token_mult": 1.0},
    {"name": "ReAct + Planning",         "success_rate": 0.85, "token_mult": 1.8},
    {"name": "ReAct + Reflection",       "success_rate": 0.90, "token_mult": 2.5},
    {"name": "Multi-Agent Ensemble",     "success_rate": 0.95, "token_mult": 4.0},
]

results = []
for profile in profiles:
    agent = SimulatedReActAgent(
        TOOL_CATALOG,
        success_rate=profile["success_rate"]
    )
    harness = EvalHarness(EVAL_TASKS)
    traces = harness.run_eval(agent)
    harness.score()

    # Adjust token counts by multiplier to simulate cost differences
    adjusted_tokens = harness.metrics_results["cost"]["total_tokens"] * profile["token_mult"]

    results.append({
        "name": profile["name"],
        "completion_rate": harness.metrics_results["task_completion"]["rate"],
        "total_tokens": adjusted_tokens,
        "cost_per_task": (adjusted_tokens / len(EVAL_TASKS)) / 1000 * 0.01,
        "efficiency": harness.metrics_results["planning"]["avg_efficiency_ratio"]
    })

# Display as table
df = pd.DataFrame(results)
df["cost_per_task_cents"] = df["cost_per_task"] * 100
print(df[["name", "completion_rate", "total_tokens", "cost_per_task_cents", "efficiency"]].to_string(index=False))

In [ ]:
# Visualize the cost-quality Pareto frontier

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Task completion rate vs total tokens (cost)
ax = axes[0]
colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6']

for i, r in enumerate(results):
    ax.scatter(r["total_tokens"], r["completion_rate"],
               s=200, c=colors[i], zorder=5, edgecolors='black', linewidths=1)
    ax.annotate(r["name"], (r["total_tokens"], r["completion_rate"]),
                textcoords="offset points", xytext=(10, 5), fontsize=8)

# Draw Pareto frontier
sorted_results = sorted(results, key=lambda x: x["total_tokens"])
pareto_x = [sorted_results[0]["total_tokens"]]
pareto_y = [sorted_results[0]["completion_rate"]]
max_rate = sorted_results[0]["completion_rate"]
for r in sorted_results[1:]:
    if r["completion_rate"] > max_rate:
        pareto_x.append(r["total_tokens"])
        pareto_y.append(r["completion_rate"])
        max_rate = r["completion_rate"]
ax.plot(pareto_x, pareto_y, 'k--', alpha=0.4, label='Pareto frontier')

ax.set_xlabel('Total Tokens Used (proxy for cost)', fontsize=11)
ax.set_ylabel('Task Completion Rate', fontsize=11)
ax.set_title('Cost-Quality Pareto Frontier', fontsize=13, fontweight='bold')
ax.set_ylim(0.3, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Marginal improvement per token
ax2 = axes[1]
sorted_by_cost = sorted(results, key=lambda x: x["total_tokens"])
names_sorted = [r["name"] for r in sorted_by_cost]
marginal_gains = []
for i in range(len(sorted_by_cost)):
    if i == 0:
        marginal_gains.append(sorted_by_cost[i]["completion_rate"])
    else:
        gain = sorted_by_cost[i]["completion_rate"] - sorted_by_cost[i-1]["completion_rate"]
        extra_tokens = sorted_by_cost[i]["total_tokens"] - sorted_by_cost[i-1]["total_tokens"]
        marginal_gains.append(gain / max(extra_tokens, 1) * 10000)  # gain per 10K tokens

bars = ax2.bar(range(len(names_sorted)), marginal_gains, color=colors)
ax2.set_xticks(range(len(names_sorted)))
ax2.set_xticklabels([n.replace(' ', '\n') for n in names_sorted], fontsize=7)
ax2.set_ylabel('Marginal Improvement\n(per 10K extra tokens)', fontsize=10)
ax2.set_title('Diminishing Returns of Spending More', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey Takeaway: There are diminishing returns to spending more tokens.")
print("The 'ReAct + Planning' approach often hits the sweet spot:")
print("  - Significantly better than basic ReAct")
print("  - Much cheaper than ensemble approaches")
print("  - Good enough for most production use cases")

### When Is a More Expensive Agent Worth It?

**Use the cheap agent when:**
- High volume, low stakes (e.g., categorizing support tickets)
- Errors can be caught downstream by humans
- Latency matters more than accuracy

**Use the expensive agent when:**
- Errors are costly (e.g., financial transactions, medical advice)
- Low volume, high stakes (e.g., code deployment, security analysis)
- The agent's output feeds into critical downstream systems

**Rule of thumb**: Calculate the **cost of a mistake** vs the **cost of the agent**. If an error costs $1000 to fix and the expensive agent prevents 10% more errors, it's worth spending 10x more on the agent for any volume > 100 tasks.

## 9. "Why Does This Work?"

### Is task completion rate enough?

**No.** Task completion rate is necessary but not sufficient. Consider:
- A coding agent that passes tests by hardcoding expected outputs -- 100% completion, 0% usefulness
- A web agent that completes purchases by buying the wrong item -- "completed" but wrong
- An RCA agent that identifies the right root cause but via flawed reasoning -- correct answer, dangerous process

Process quality matters because:
1. **Safety**: We need to trust HOW the agent arrives at answers, not just THAT it does
2. **Generalization**: Good process generalizes; lucky guesses don't
3. **Debugging**: When an agent fails, process traces help you fix it

### How do you evaluate an agent on novel tasks?

The tension is **generalization vs memorization**. SWE-bench tasks are public; models may have seen them in training. Solutions:
- **Held-out test sets**: Never release the full benchmark (GAIA does this)
- **Contamination checks**: Test on tasks created after model training cutoff
- **Process-based evaluation**: Grade the trajectory, not just the answer
- **Dynamic benchmarks**: Generate new tasks programmatically (e.g., new GitHub issues)

### What makes a good agent benchmark?

Criteria:
1. **Realistic tasks**: Should reflect actual agent use cases, not toy problems
2. **Verifiable answers**: Must be automatically checkable (not subjective)
3. **Diverse difficulty**: Easy to hard, so you can see the capability curve
4. **Resistant to memorization**: Hard to solve by recalling training data
5. **Reproducible**: Same setup yields same results (modulo model non-determinism)
6. **Multi-dimensional**: Measures cost, latency, and safety -- not just accuracy

---
## Interview Question Bank

*Evaluation questions reveal whether a candidate has actually shipped an agent vs. only built demos. Anyone can build a demo that works on 3 examples. Evaluation is how you know it works on 3,000.*

---

### Q1: "How would you evaluate an agent system before deploying to production?" -- SYSTEM DESIGN (30 min)

**What this tests**: Do you have an evaluation methodology, or do you just vibe-check?

**Good answer** (hire):
- Task completion rate on a held-out test set
- Accuracy metrics (did the agent produce the right answer?)
- Latency and cost per task

**Great answer** (strong hire): Describes a staged evaluation pipeline:

1. **Unit tests**: Does each tool work correctly in isolation? Does the prompt produce expected outputs on known inputs?
2. **Integration tests**: Does the agent complete short (2-3 step) tasks reliably? Test the full loop: LLM call -> tool use -> observation -> next step.
3. **Shadow mode**: Run the new agent in parallel with the existing system (or human baseline). Compare outputs without serving the new agent to users.
4. **Canary deployment**: Route 5% of traffic to the new agent. Monitor task completion, error rate, cost, latency, and user feedback.
5. **Full deployment**: Gradually ramp from 5% to 100% with automated rollback triggers.

Additionally discusses:
- **Safety evaluation**: Does the agent ever take harmful actions? Run adversarial inputs specifically designed to trigger unsafe behavior.
- **Cost monitoring**: Track token usage per session. Set alerts for sessions that exceed 2x the median cost.
- **Latency SLOs**: Users expect a response in <5 seconds for simple tasks, <30 seconds for complex ones. Measure P50, P95, P99.

**Red flag**: Only mentions "accuracy on a benchmark." No staged rollout. No safety evaluation. No cost awareness.

**Follow-up**: "Your agent passes all benchmarks but users complain it is unhelpful. Diagnose."
- Good: look at user feedback, analyze failure cases
- Great: discusses the gap between benchmarks and real-world use. Benchmarks test *can the agent complete a well-specified task?* Users often have *underspecified* requests. The agent might be failing at clarification, not execution. Also: benchmarks do not test *how* the agent communicates -- an agent that solves the task but is confusing or verbose will get complaints.

---

### Q2: "SWE-bench -- what does it measure and what does it not?"

**What this tests**: Critical thinking about benchmarks. Every candidate cites SWE-bench. Few understand its limitations.

**Good answer**: SWE-bench measures an agent's ability to resolve real GitHub issues by producing correct patches.

**Great answer**: Discusses both strengths and limitations:

**What it measures well**:
- End-to-end coding ability (read issue, understand codebase, write patch, pass tests)
- Realistic task distribution (real bugs from real repos)
- Verifiable correctness (existing test suites)

**What it does NOT measure**:
- Only Python repositories (does not generalize to other languages)
- Only well-specified issues with existing test suites (real issues are often vague)
- No design or architecture tasks (only bug fixes and small features)
- No communication (real SWE work involves clarifying requirements, code review, documentation)
- **Contamination risk**: the issues are public on GitHub. Models may have seen them in training data. SWE-bench Verified partially addresses this but the concern remains.
- **Why real-world coding is harder**: Real issues require understanding organizational context, unwritten conventions, and business logic that is not in the codebase.

**Red flag**: Cites SWE-bench numbers without any caveats. Cannot name a limitation.

---

### Q3: "Design an evaluation harness for a customer support agent" -- APPLIED (20 min)

**What this tests**: Can you apply evaluation principles to a specific domain?

**Good answer**: Defines metrics (resolution rate, customer satisfaction, escalation rate) and a test set of representative tickets.

**Great answer**:
- **Metrics**: resolution rate, first-response relevance, customer satisfaction (CSAT), escalation rate, cost per ticket, time to resolution
- **Test set design**: stratified by difficulty (easy/medium/hard), category (billing/technical/account), and edge cases (angry customer, ambiguous request, multi-issue ticket)
- **LLM-as-judge**: use a stronger model to evaluate response quality on a rubric (empathy, accuracy, completeness)
- **A/B testing**: compare agent responses to human agent responses on the same tickets
- **Safety tests**: ensure the agent never leaks PII, never makes unauthorized refunds, never promises things it cannot deliver

---
## Production Implementation Notes

*Evaluation is not a one-time activity. In production, evaluation is continuous monitoring.*

### The Production Evaluation Stack

```
Level 1: Offline Eval (before deployment)
  - Benchmark suites (task completion, accuracy)
  - Adversarial testing (safety, edge cases)
  - Cost profiling (median and P99 cost per session)

Level 2: Shadow Eval (before serving users)
  - Run new agent in parallel with production
  - Compare outputs (automated diff + human spot-check)
  - Measure latency impact

Level 3: Online Eval (in production)
  - A/B testing with real users
  - Real-time monitoring dashboards
  - Automated alerts for anomalies (cost spike, error rate spike)

Level 4: Continuous Eval (ongoing)
  - Weekly regression testing on expanding benchmark
  - Monthly human evaluation of sampled sessions
  - Quarterly safety audit
```

### Metrics That Matter in Production

| Metric | What It Catches | Alert Threshold |
|--------|----------------|-----------------|
| **Task completion rate** | Agent getting worse overall | Drop >5% week-over-week |
| **P95 latency** | Slow sessions frustrating users | >30s for simple tasks |
| **Cost per session** | Budget overruns | >2x median |
| **Error rate** | Tool failures, API issues | >10% of sessions |
| **Escalation rate** | Agent failing and needing human | >30% of sessions |
| **Safety incident rate** | Agent taking harmful actions | Any incident = immediate review |

### The Eval Gap Nobody Talks About

The biggest gap in agent evaluation today: **there is no good way to evaluate partially correct outputs.**

- Binary metrics (pass/fail) miss the difference between "completely wrong" and "90% right with one small error"
- LLM-as-judge can assess quality but is expensive and has its own biases (tends to prefer verbose outputs)
- Human evaluation is the gold standard but does not scale

The research frontier is developing reliable, cheap, automatic evaluation methods that capture partial credit. This is an excellent topic to bring up in a research discussion interview -- it shows you understand what is actually hard.

---
## How This Gets Tested in Interviews

### The Evaluation Interview Is a Maturity Test

Evaluation questions are used specifically to distinguish between candidates who have shipped vs. those who have only prototyped. The depth of your evaluation methodology directly correlates with your production experience.

### What Interviewers Are Really Asking

When they ask "how would you evaluate this agent?", they are testing:

1. **Do you think about evaluation BEFORE building?** Senior engineers define success metrics first, then build. Junior engineers build first, then figure out how to evaluate. If you immediately start discussing the agent architecture instead of evaluation criteria, that is a signal.

2. **Do you know what "good" looks like?** Can you set concrete thresholds? "90% task completion on our benchmark" is better than "high accuracy." "P95 latency under 10 seconds" is better than "fast."

3. **Can you think adversarially?** A strong candidate immediately asks: "What could go wrong? What inputs would break this?" This is the safety evaluation mindset.

### The "Benchmark vs. Real World" Discussion

This comes up in nearly every senior-level interview. The interviewer will describe a benchmark result and ask you to critique it. The correct response is always nuanced:

> "Benchmark X shows Y performance, which is encouraging, but I would be cautious about generalizing because [specific limitation]. In production, I would expect [lower/different] performance because [real-world complexity]. I would validate with [specific evaluation plan]."

Never say "benchmarks are useless" (they are not -- they are a necessary first filter). Never say "our benchmark results prove it works" (they do not -- they prove it works on the benchmark).

### Red Flags Interviewers Watch For

- **No mention of cost**: If you design an evaluation that would cost $10,000 to run once, and you do not mention this, you have never run an evaluation at scale
- **No mention of statistical significance**: "We ran it on 10 examples and got 80%" -- that confidence interval is enormous. You need 100+ examples minimum for reliable estimates.
- **Evaluating only the happy path**: If your test set does not include adversarial inputs, edge cases, and failure modes, it is not a real evaluation
- **No baseline comparison**: "Our agent gets 75%" means nothing without "compared to X which gets Y%"

## 10. Flashcard Summary

Study these for interview prep. Cover the answer, try to recall, then check.

---

**Q1**: What are the six core metrics for agent evaluation?

**A1**: (1) Task completion rate, (2) Tool use accuracy, (3) Planning efficiency, (4) Cost (tokens), (5) Error recovery rate, (6) Time to completion.

---

**Q2**: Why is agent evaluation harder than LLM evaluation?

**A2**: Agents have multi-step trajectories (not single outputs), multiple valid solution paths, non-deterministic tool interactions, and cost/safety dimensions that pure text evaluation lacks.

---

**Q3**: What does SWE-bench evaluate?

**A3**: Whether an agent can resolve real GitHub issues by editing code in a full repository. SWE-bench Verified is a 500-instance human-validated subset curated by OpenAI (Aug 2024). Scores went ~33% (late 2024) → ~70% (mid-2025) → ~80% (early 2026); OpenAI deprecated it as an internal eval in Feb 2026 over test flaws and contamination.

---

**Q4**: What does WebArena evaluate?

**A4**: Whether an agent can complete tasks on real self-hosted websites (shopping, Reddit, GitLab), testing navigation, form filling, and multi-step web interactions.

---

**Q5**: What is the Pareto frontier in agent evaluation?

**A5**: The set of agents where you cannot improve one metric (e.g., accuracy) without worsening another (e.g., cost). Agents on the frontier represent optimal tradeoffs.

---

**Q6**: Name three categories of agent failure.

**A6**: Tool selection error (wrong tool), argument error (right tool, wrong inputs), planning error (wrong step sequence), hallucination (fabricated info), context limit (lost track of info), unrecovered tool failure.

---

**Q7**: Why is process quality important beyond task completion?

**A7**: For safety (we need to trust HOW agents arrive at answers), generalization (good process transfers; lucky guesses don't), and debugging (traces help fix failures).

---

**Q8**: What does BFCL (Berkeley Function-Calling Leaderboard) measure?

**A8**: Raw tool/function calling accuracy across simple, parallel, multiple, and nested scenarios. It tests the foundation of agent capability -- can the model produce correct tool calls?

---

**Q9**: What is the cost of a mistake framework?

**A9**: Compare the cost of agent errors (business impact of wrong answers) vs the marginal cost of a better agent. If mistake costs >> agent costs, invest in accuracy. For low-stakes tasks, use cheaper agents.

---

**Q10**: How do you handle non-determinism in agent evaluation?

**A10**: Run multiple trials per task, report mean and standard deviation, use statistical tests for agent comparisons, and set random seeds where possible.

---

**Q11**: What is tau-bench and why is it important?

**A11**: tau-bench evaluates agents on enterprise tasks (airline, retail customer service). It tests the full production stack: tool use, policy compliance, multi-turn state tracking, and real-world business logic.

---

**Q12**: What makes a good agent benchmark?

**A12**: (1) Realistic tasks reflecting actual use cases, (2) automatically verifiable answers, (3) diverse difficulty levels, (4) resistance to memorization, (5) reproducibility, (6) multi-dimensional (measures cost, latency, safety, not just accuracy).

## 11. Interview Talking Points

### How to discuss agent evaluation in a frontier lab interview:

**Opening frame**: "Agent evaluation is a multi-objective optimization problem. Unlike LLM eval where you compare text outputs, agent eval must consider the entire trajectory: was the right tool called, was the reasoning sound, was the cost reasonable, and did the agent recover from failures?"

**Connect to evaluation design**: "In a multi-agent root-cause-analysis system, evaluation must check not just whether agents identify the correct root cause, but whether their diagnostic process is sound -- an agent that gets lucky on one failure mode won't generalize. A strong evaluation harness tracks tool selection accuracy, reasoning chain quality, and cost per diagnosis."

**Show benchmark awareness**: "I stay current with the benchmark landscape -- SWE-bench Verified for coding agents, WebArena for web tasks, GAIA for general assistance, tau-bench for enterprise scenarios. Each has strengths and blind spots. For instance, SWE-bench is Python-only and doesn't test the agent's ability to understand user intent -- just to fix a known bug."

**Demonstrate depth**: "A key insight is that the cost-quality Pareto frontier is more informative than any single accuracy number. A 95% accurate agent that costs 4x more than a 90% accurate one is only worth it if mistakes cost more than 20x the per-task agent cost. This kind of economic analysis is often missing from academic benchmarks but essential for production systems."

**Forward-looking**: "I think the next frontier in agent eval is process-based evaluation -- grading the reasoning and tool use, not just the final answer. This connects directly to alignment: we want agents that solve problems for the right reasons, not ones that happen to get lucky."

---

*End of Notebook 15*